# Swiss parliamentary source import

This notebook recreates the country-neutral SQLite research database exclusively from the committed schema and verified local source manifest. It performs no network requests and no political classification.

## Locate the repository and load the transparent import helper

The search walks upward from the kernel's current directory, so execution works from the repository root or the `notebooks` directory without hard-coded machine paths.

In [1]:
from pathlib import Path
import json
import sqlite3
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
repository_root = next(
    (path for path in candidates if (path / 'database/schema.sql').is_file() and (path / '.agents/CONTEXT.md').is_file()),
    None,
)
if repository_root is None:
    raise RuntimeError('Repository root not found above the current working directory')
sys.path.insert(0, str(repository_root / 'src'))

from politiks.importer import query_rows, recreate_database

database_path = repository_root / 'database/parliament.sqlite'
manifest_path = repository_root / 'source/manifests/fixture.jsonl'
print('Repository root: located')
print(f'Local manifest: {manifest_path.relative_to(repository_root).as_posix()}')
print(f'Generated database: {database_path.relative_to(repository_root).as_posix()}')

Repository root: located
Local manifest: source/manifests/fixture.jsonl
Generated database: database/parliament.sqlite


## Recreate and import the database

The helper deletes only the generated SQLite file, applies `database/schema.sql`, verifies every source byte count and SHA256 against the manifest, and imports all supported JSON shapes in one transaction. Non-JSON documentation is still registered for provenance.

In [2]:
report = recreate_database(
    repository_root, manifest_path=manifest_path, database_path=database_path
)
print(json.dumps({
    'snapshot': report.snapshot_name,
    'source_files': report.source_files,
    'source_records': report.source_records,
    'normalized_files': report.normalized_files,
    'registered_only': len(report.skipped_files),
}, ensure_ascii=False, indent=2))

{
  "snapshot": "fixture",
  "source_files": 38,
  "source_records": 960,
  "normalized_files": 31,
  "registered_only": 7
}


## Inspect bounded logical row counts

These counts make repeated imports comparable and show the boundary between provenance records, parliamentary entities, and individual voting choices.

In [3]:
import pandas as pd
from IPython.display import display

display(pd.DataFrame(
    [{'table': table, 'rows': rows} for table, rows in report.table_counts.items()]
).style.hide(axis='index'))
display(pd.DataFrame(
    [{'recorded_choice': choice, 'rows': rows} for choice, rows in report.choice_counts.items()]
).style.hide(axis='index'))

table,rows
source_file,38
source_record,960
country,1
legislature,1
chamber,3
legislative_period,16
parliamentary_session,22
subdivision,26
committee,50
political_party,52


recorded_choice,rows
abstain,42
no,781
not_participating,87
presiding,10
yes,1321


## Report coverage and unresolved links explicitly

The sampled vote payloads omit a chamber field. They remain labeled as unresolved instead of being silently assigned from their roughly 200-person shape. Membership gaps are likewise counted; a small number of profile-derived intervals are retained but marked as inferred.

In [4]:
display(pd.DataFrame(report.chamber_year_counts).style.hide(axis='index'))
limitations = {
    'voting events with unresolved chamber': report.unresolved_event_chambers,
    'voting events without an affair': report.unlinked_event_matters,
    'choices without date-valid party membership': report.choices_without_dated_party,
    'choices without date-valid faction membership': report.choices_without_dated_faction,
}
print(json.dumps(limitations, ensure_ascii=False, indent=2))

chamber,year,voting_events
Nicht aufgelöst,2017,1
Nicht aufgelöst,2020,20
Nicht aufgelöst,2021,5
Nicht aufgelöst,2022,4
Nicht aufgelöst,2023,21
Nicht aufgelöst,2025,2
Nicht aufgelöst,2026,1


{
  "voting events with unresolved chamber": 54,
  "voting events without an affair": 0,
  "choices without date-valid party membership": 2169,
  "choices without date-valid faction membership": 2158
}


## Verify a representative evidence join

This query follows one choice through its voting event and affair to the person and the party/faction interval covering the vote date. The `is_inferred` and `evidence_basis` columns prevent current-profile associations from masquerading as explicit historical membership.

In [5]:
representative_join = query_rows(database_path, '''
SELECT ve.source_identifier AS vote_id, ve.occurred_at, pm.formatted_identifier AS affair_id,
       pm.title, p.display_name, vc.raw_decision, vc.normalized_choice,
       pp.abbreviation AS party, pf.abbreviation AS faction,
       ppm.is_inferred AS party_is_inferred, ppm.evidence_basis AS party_basis
FROM voting_choice vc
JOIN voting_event ve ON ve.id = vc.voting_event_id
JOIN parliamentary_matter pm ON pm.id = ve.matter_id
JOIN person p ON p.id = vc.person_id
JOIN person_party_membership ppm ON ppm.person_id = p.id
 AND (ppm.date_from IS NULL OR ppm.date_from <= substr(ve.occurred_at, 1, 10))
 AND (ppm.date_to IS NULL OR ppm.date_to >= substr(ve.occurred_at, 1, 10))
JOIN political_party pp ON pp.id = ppm.party_id
JOIN person_faction_membership pfm ON pfm.person_id = p.id
 AND (pfm.date_from IS NULL OR pfm.date_from <= substr(ve.occurred_at, 1, 10))
 AND (pfm.date_to IS NULL OR pfm.date_to >= substr(ve.occurred_at, 1, 10))
JOIN parliamentary_faction pf ON pf.id = pfm.faction_id
WHERE p.display_name = 'Thomas Aeschi'
ORDER BY ve.occurred_at DESC LIMIT 3
''')
display(pd.DataFrame(representative_join).style.hide(axis='index'))

vote_id,occurred_at,affair_id,title,display_name,raw_decision,normalized_choice,party,faction,party_is_inferred,party_basis
35937,2026-03-20T08:12:46Z,15.320,Systematische Vorlage des Strafregisterauszugs bei der Beantragung von Aufenthaltsbewilligungen durch EU-Bürgerinnen und -Bürger (1),Thomas Aeschi,Yes,yes,SVP,V,1,inferred_current_profile_party_over_mandate
35025,2025-09-26T08:03:37Z,12.409,Entschädigung von Hilfeleistungen von Angehörigen im Rahmen des Assistenzbeitrages,Thomas Aeschi,No,no,SVP,V,1,inferred_current_profile_party_over_mandate
34192,2025-03-21T09:55:00Z,15.320,Systematische Vorlage des Strafregisterauszugs bei der Beantragung von Aufenthaltsbewilligungen durch EU-Bürgerinnen und -Bürger (1),Thomas Aeschi,Yes,yes,SVP,V,1,inferred_current_profile_party_over_mandate


## Enforce final integrity and identifier checks

The final cell fails execution on foreign-key violations, duplicate stable identifiers, missing source files, or an empty representative join. Passing it makes the notebook an executable import contract rather than a narrative-only artifact.

In [6]:
checks = query_rows(database_path, '''
SELECT
  (SELECT COUNT(*) FROM pragma_foreign_key_check) AS foreign_key_violations,
  (SELECT COUNT(*) FROM (
     SELECT source_system, namespace, identifier FROM person_identifier
     GROUP BY source_system, namespace, identifier HAVING COUNT(*) > 1
   )) AS duplicate_person_identifiers,
  (SELECT COUNT(*) FROM (
     SELECT source_system, source_identifier FROM voting_event
     GROUP BY source_system, source_identifier HAVING COUNT(*) > 1
   )) AS duplicate_voting_identifiers,
  (SELECT COUNT(*) FROM source_file) AS source_files
''')[0]
assert checks['foreign_key_violations'] == 0
assert checks['duplicate_person_identifiers'] == 0
assert checks['duplicate_voting_identifiers'] == 0
assert checks['source_files'] == report.source_files == 38
assert representative_join
print(json.dumps(checks, ensure_ascii=False, indent=2))
print('Import and integrity checks passed.')

{
  "foreign_key_violations": 0,
  "duplicate_person_identifiers": 0,
  "duplicate_voting_identifiers": 0,
  "source_files": 38
}
Import and integrity checks passed.
